In [1]:
# ============================================================
# CARGAR DATASET
# ============================================================

import pandas as pd

df_model = pd.read_csv("dataset_limpio_churn.csv")

print(df_model.shape)

(317579, 35)


In [2]:
df_model = pd.read_csv("dataset_limpio_churn.csv")

print(df_model.columns.tolist())

['cliente_id', 'fecha', 'zona_id_x', 'tipo_plan_x', 'num_lineas_x', 'cargo_base', 'consumo_extra', 'descuento_aplicado', 'importe_total', 'dias_retraso_pago', 'impago_flag', 'variacion_consumo_pct', 'stress_calidad_lag', 'incidencia_masiva_lag', 'churn', 'zona_id_y', 'region', 'tipo_zona', 'poblacion_zona', 'edad', 'sexo', 'estado_civil', 'num_lineas_y', 'tipo_plan_y', 'tipo_dispositivo', 'ingreso_estimado', 'antiguedad_meses', 'descuento_activo', 'factura_mes_anterior', 'subida_factura', 'consumo_mes_anterior', 'cambio_consumo', 'impago_mes_anterior', 'retraso_mes_anterior', 'alerta_churn']


In [6]:
# ============================================================
# MODELO CHURN MEJORADO REAL
# RANDOM FOREST + SMOTE — CORREGIDO PARA TU DATASET
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

from imblearn.over_sampling import SMOTE


# ============================================================
# CARGAR DATASET
# ============================================================

df_model = pd.read_csv("dataset_limpio_churn.csv")

print("Shape inicial:", df_model.shape)


# ============================================================
# FECHAS Y ORDEN TEMPORAL
# ============================================================

df_model["fecha"] = pd.to_datetime(
    df_model["fecha"],
    errors="coerce"
)

df_model = df_model.sort_values(
    ["cliente_id", "fecha"]
)


# ============================================================
# CREAR VARIABLES TEMPORALES
# ============================================================

df_model["media_importe_3m"] = (
    df_model
    .groupby("cliente_id")["importe_total"]
    .transform(
        lambda x: x.rolling(
            3,
            min_periods=1
        ).mean()
    )
)

df_model["media_retraso_3m"] = (
    df_model
    .groupby("cliente_id")["dias_retraso_pago"]
    .transform(
        lambda x: x.rolling(
            3,
            min_periods=1
        ).mean()
    )
)

df_model["impagos_3m"] = (
    df_model
    .groupby("cliente_id")["impago_flag"]
    .transform(
        lambda x: x.rolling(
            3,
            min_periods=1
        ).sum()
    )
)


# ============================================================
# VARIABLES
# ============================================================

variables = [

    "edad",
    "antiguedad_meses",
    "ingreso_estimado",
    "num_lineas_x",
    "descuento_activo",

    "importe_total",
    "dias_retraso_pago",
    "impago_flag",

    "factura_mes_anterior",
    "consumo_mes_anterior",
    "retraso_mes_anterior",
    "impago_mes_anterior",

    "media_importe_3m",
    "media_retraso_3m",
    "impagos_3m",

    "cambio_consumo",
    "subida_factura",
    "variacion_consumo_pct",

    "stress_calidad_lag",
    "incidencia_masiva_lag"

]


# ============================================================
# ELIMINAR NULLS
# ============================================================

df_model = df_model.dropna(
    subset=variables + ["churn"]
)


# ============================================================
# X / y
# ============================================================

X = df_model[variables]
y = df_model["churn"]


# ============================================================
# LIMPIEZA
# ============================================================

X = X.replace(
    [np.inf, -np.inf],
    np.nan
)

X = X.fillna(
    X.median(numeric_only=True)
)

X = X.fillna(0)


# ============================================================
# TRAIN TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y

)


# ============================================================
# SMOTE SOLO TRAIN
# ============================================================

smote = SMOTE(
    sampling_strategy=0.50,
    random_state=42,
    k_neighbors=5
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)


# ============================================================
# DISTRIBUCIÓN
# ============================================================

print("="*60)
print("DISTRIBUCIÓN TRAIN DESPUÉS DE SMOTE")
print("="*60)

print(y_train_smote.value_counts())


# ============================================================
# MODELO RANDOM FOREST
# ============================================================

modelo = RandomForestClassifier(

    n_estimators=400,
    max_depth=10,
    min_samples_split=15,
    min_samples_leaf=8,

    class_weight="balanced",

    random_state=42,
    n_jobs=-1

)


# ============================================================
# ENTRENAMIENTO
# ============================================================

modelo.fit(
    X_train_smote,
    y_train_smote
)


# ============================================================
# PROBABILIDADES
# ============================================================

y_prob = modelo.predict_proba(X_test)[:, 1]


# ============================================================
# THRESHOLD
# ============================================================

threshold = 0.35

y_pred = (y_prob >= threshold).astype(int)


# ============================================================
# MÉTRICAS
# ============================================================

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

roc = roc_auc_score(
    y_test,
    y_prob
)


# ============================================================
# RESULTADOS
# ============================================================

print("\n" + "="*60)
print("RESULTADOS MODELO FINAL")
print("="*60)

print(f"\nAccuracy: {accuracy:.2%}")
print(f"Precision churn: {precision:.4f}")
print(f"Recall churn: {recall:.4f}")
print(f"F1 churn: {f1:.4f}")
print(f"ROC-AUC: {roc:.4f}")


# ============================================================
# MATRIZ CONFUSIÓN
# ============================================================

matriz = confusion_matrix(
    y_test,
    y_pred
)

print("\nMatriz de confusión:")
print(matriz)

tn, fp, fn, tp = matriz.ravel()

print("\nClientes churn reales:", tp + fn)
print("Clientes churn detectados:", tp)
print("Clientes churn no detectados:", fn)
print("Falsos positivos:", fp)
print("Total predichos como churn:", tp + fp)


# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)


# ============================================================
# IMPORTANCIA VARIABLES
# ============================================================

importancias = pd.DataFrame({

    "Variable": variables,
    "Importancia": modelo.feature_importances_

})

importancias = importancias.sort_values(
    by="Importancia",
    ascending=False
)


print("\n" + "="*60)
print("IMPORTANCIA VARIABLES")
print("="*60)

print(importancias.head(15))

Shape inicial: (317579, 35)
DISTRIBUCIÓN TRAIN DESPUÉS DE SMOTE
churn
0    244503
1    122251
Name: count, dtype: int64

RESULTADOS MODELO FINAL

Accuracy: 83.91%
Precision churn: 0.0132
Recall churn: 0.3308
F1 churn: 0.0254
ROC-AUC: 0.6393

Matriz de confusión:
[[51488  9638]
 [  261   129]]

Clientes churn reales: 390
Clientes churn detectados: 129
Clientes churn no detectados: 261
Falsos positivos: 9638
Total predichos como churn: 9767

Classification Report:

              precision    recall  f1-score   support

           0       0.99      0.84      0.91     61126
           1       0.01      0.33      0.03       390

    accuracy                           0.84     61516
   macro avg       0.50      0.59      0.47     61516
weighted avg       0.99      0.84      0.91     61516


IMPORTANCIA VARIABLES
                 Variable  Importancia
14             impagos_3m     0.285598
13       media_retraso_3m     0.199944
11    impago_mes_anterior     0.105373
6       dias_retraso_pago 

El modelo Random Forest combinado con SMOTE permitió abordar el fuerte desbalanceo existente en los datos y consiguió identificar aproximadamente un 33 % de los clientes que realmente abandonan la compañía (recall = 0,3308). Sin embargo, la precisión obtenida fue baja (0,0132), lo que implica un elevado número de falsos positivos. El modelo alcanzó una exactitud global del 83,91 % y un ROC-AUC de 0,6393, mostrando una capacidad discriminativa moderada. Las variables con mayor importancia estuvieron relacionadas con la calidad del servicio, el historial de impagos y las variaciones en el consumo y la facturación. En conjunto, aunque el modelo es capaz de detectar parte de los clientes con riesgo de abandono, sus resultados son mejorables y sugieren la conveniencia de utilizar modelos más avanzados, como XGBoost, para obtener una mayor capacidad predictiva.
Por ello, este modelo se consideró una alternativa válida, pero no fue seleccionado como modelo final debido a que otros algoritmos, especialmente XGBoost, proporcionaron un mejor equilibrio entre capacidad de detección y rendimiento global.